### 1. Generative AI Model Selection & Setup

**Model Selected:** LLaMA 3.1 (8B) via Groq API.


**Rationale:** We opted to explore robust open-source models as suggested in the project guidelines. LLaMA 3.1 is a highly capable and efficient generative AI model. We integrated it via the Groq API because of its exceptionally high inference speed (LPU technology), making it ideal for real-time e-commerce analytics and generating instant marketing insights without latency.


In [1]:
# Install the Groq library
!pip install groq -q

# Import required libraries
import os
import pandas as pd
from groq import Groq
from google.colab import userdata

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 3.4 MB/s eta 0:00:00


###2. Prompt Template Design Documentation

### **Template 1**

**Template Name & ID:** Basic Prediction Explanation (T1)

**Intended Use Case:**

This template is used to explain the model prediction in a simple and clear way for non-technical users.

**Design Rationale (reason for choosing this approach):**

This approach was chosen to make the system understandable for users who do not have a technical background. It focuses on clarity and highlights the main factors influencing the prediction.

**Prompt Structure (with placeholders):**
You are an e-commerce analytics assistant.

**User session features:**
{features}

**Model prediction:**
{prediction}

Explain the prediction in simple and clear language.
Mention the main factors affecting the result.

**Example Input/Output:**

**Input:**

features = PageValues: 0, BounceRates: 0.20, ExitRates: 0.20, VisitorType: Returning Visitor
prediction = No Purchase

**Output:**

The model predicts that the user is unlikely to make a purchase. This is mainly because the bounce rate and exit rate are high, indicating low engagement. Also, the PageValues score is zero, which suggests that the visited pages were not strongly related to purchases.

**Assumptions & Limitations:**

This template assumes that the model prediction is correct. It provides general explanations but does not include detailed recommendations or personalized insights.

### **Template 2**

**Template Name & ID:** Marketing Recommendation Generator (T2)

**Intended Use Case:**

This template is used to generate a suitable marketing action based on the model prediction.

**Design Rationale (reason for choosing this approach):**

This approach connects machine learning outputs with business applications by transforming predictions into actionable recommendations that can improve decision-making.

**Prompt Structure (with placeholders):**
You are a digital marketing assistant.

**User session features:**
{features}

**Model prediction:**
{prediction}

Based on this prediction, suggest one suitable marketing action.
Explain why this action is appropriate.

**Example Input/Output:**

**Input:**

features = ProductRelated: 10, ProductRelated_Duration: 627.5, VisitorType: Returning Visitor
prediction = No Purchase

**Output:**

A suitable action is to offer a limited-time discount or reminder. The user spent time browsing products but did not complete a purchase, which suggests interest without commitment. A small incentive may increase the likelihood of conversion.

**Assumptions & Limitations:**

This template assumes that marketing strategies such as discounts or reminders are available. It does not guarantee that the recommendation will lead to a successful purchase.

### **Template 3**

**Template Name & ID:** Cluster-Based Personalized Explanation (T3)

**Intended Use Case:**

This template is used to provide personalized explanations by combining the model prediction with user behavior grouping results.

**Design Rationale (reason for choosing this approach):**

This approach was chosen to integrate clustering results into the system to better understand user behavior. By grouping users based on behavior patterns (such as engagement level and browsing duration), the system can provide more personalized and context-aware explanations. This improves the quality of insights compared to using prediction alone.

**Prompt Structure (with placeholders):**
You are an AI assistant for an e-commerce analytics system.

**User session features:**
{features}

**Model prediction:**
{prediction}

**User cluster information:**
Cluster ID: {cluster_id}
Cluster description: {cluster_description}

Explain the prediction and relate it to the user's cluster.
Describe how users in this cluster typically behave.
Keep the explanation simple and clear.

**Example Input/Output:**

**Input:**

features = PageValues: 35.4, BounceRates: 0.01, ExitRates: 0.03
prediction = Purchase
cluster_id = 1
cluster_description = High engagement users with low bounce rates and high page values

**Output:**

The model predicts that the user is likely to make a purchase. This is supported by strong engagement indicators such as low bounce and exit rates, along with a high PageValues score. Additionally, the user belongs to a cluster of highly engaged users who typically spend more time browsing and are more likely to complete purchases. This behavior pattern increases the likelihood of conversion.

**Assumptions & Limitations:**

This template assumes that clustering results are meaningful and accurately represent user behavior patterns. If the clustering quality is weak or not well-defined, the explanation may not add significant value. It also depends only on session data and may not capture all external factors.

### **Template 4**

**Template Name & ID**: Safe & Responsible Explanation (T4)

**Intended Use Case:**

This template is used to provide cautious explanations that avoid overconfidence in model predictions.

**Design Rationale (reason for choosing this approach):**

This approach ensures responsible AI usage by presenting predictions as probabilistic rather than certain. It helps improve user trust and avoids misleading conclusions.

**Prompt Structure (with placeholders):**
You are a responsible AI assistant.

**User session features:**
{features}

**Model prediction:**
{prediction}

**Model confidence:**
{confidence}

Explain the prediction carefully.
Do not claim certainty.
Mention one limitation.
Explain the prediction based only on the given data.

**Example Input/Output:**

**Input:**

features = PageValues: 0, BounceRates: 0.20
prediction = No Purchase
confidence = 0.87

**Output:**

The model suggests that the user is unlikely to make a purchase based on the available session data. This prediction is influenced by low engagement indicators such as high bounce rate. However, this result is not certain because it does not consider all possible factors such as user preferences or external conditions.

**Assumptions & Limitations:**

 This template assumes that model confidence scores are available. It only considers session-based data and may not reflect all real-world factors.

In [2]:
# ==========================================
# Prompt Templates Functions (Code)
# ==========================================

# Template 1: Basic Prediction Explanation
def get_prompt_t1(features, prediction):
    return f"""You are an e-commerce analytics assistant.
User session features: {features}
Model prediction: {prediction}
Explain the prediction in simple and clear language. Mention the main factors affecting the result."""

# Template 2: Marketing Recommendation Generator
def get_prompt_t2(features, prediction):
    return f"""You are a digital marketing assistant.
User session features: {features}
Model prediction: {prediction}
Based on this prediction, suggest one suitable marketing action. Explain why this action is appropriate."""

# Template 3: Cluster-Based Personalized Explanation
def get_prompt_t3(features, prediction, cluster_id, cluster_description):
    return f"""You are an AI assistant for an e-commerce analytics system.
User session features: {features}
Model prediction: {prediction}
User cluster information: Cluster ID: {cluster_id} | Cluster description: {cluster_description}
Explain the prediction and relate it to the user's cluster. Describe how users in this cluster typically behave. Keep the explanation simple and clear."""

# Template 4: Safe & Responsible Explanation
def get_prompt_t4(features, prediction, confidence):
    return f"""You are a responsible AI assistant.
User session features: {features}
Model prediction: {prediction}
Model confidence: {confidence}
Explain the prediction carefully. Do not claim certainty. Mention one limitation. Explain the prediction based only on the given data."""

print("✅ تم تحويل القوالب الأربعة إلى دوال برمجية بنجاح!")

✅ تم تحويل القوالب الأربعة إلى دوال برمجية بنجاح!


### 3. Implementation & API Integration Code (Safe Key Handling)
*Note: The API key is securely retrieved using Google Colab's `userdata` secrets management. It is never hardcoded in the script to ensure security compliance.*

In [3]:
# ==========================================
# Safe Key Handling & API Integration
# ==========================================
import os
from groq import Groq
from google.colab import userdata

try:
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    os.environ["GROQ_API_KEY"] = GROQ_API_KEY
    client = Groq()
    MODEL_NAME = "llama-3.1-8b-instant"
    print("✅ Successfully connected to Groq API!")
except Exception as e:
    print(f"❌ Error: {e}")

def generate_ai_response(prompt_text):
    try:
        response = client.chat.completions.create(
            messages=[
                # هنا أضفنا ملاحظة الـ Scaling للنموذج عشان يفهم الأرقام صح
                {"role": "system", "content": "You are an expert e-commerce AI assistant. Note: The input features are standardized (Z-score scaled). A negative value means it is below the average, and a positive value means it is above the average. Explain this clearly without confusion."},
                {"role": "user", "content": prompt_text}
            ],
            model=MODEL_NAME,
            temperature=0.7,
            max_tokens=800  # ⬅️ هنا زدنا المساحة عشان ما ينقطع الكلام
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"Error: {e}"

❌ Error: Secret GROQ_API_KEY does not exist.


### 4. Testing Framework & Output Comparison


In [4]:
# ==========================================
# Testing Framework & Output Comparison
# ==========================================
import pandas as pd

try:
    # 1. Read the finalized clustered dataset
    df = pd.read_csv('clustered_data.csv')

    # 2. Sample 3 users for testing
    test_cases = df.sample(n=3, random_state=42)

    # List to store results for the quantitative analysis later
    generated_results = []

    # 3. Loop through the test cases and generate outputs
    for index, row in test_cases.iterrows():
        features = f"PageValues: {row.get('PageValues', 0):.2f}, BounceRates: {row.get('BounceRates', 0):.2f}"
        prediction = "Purchase" if row.get('Revenue', 0) == 1 else "No Purchase"
        cluster_id = row.get('Cluster', 0)

        # Determine cluster description based on your model
        cluster_desc = "Engaged Browsers" if cluster_id == 0 else "Traffic-Driven"
        confidence = "85%"

        print(f"\n{'='*70}")
        print(f"👤 TEST CASE (Visitor Index: {index}) | Prediction: {prediction} | Cluster: {cluster_id}")
        print(f"{'='*70}")

        # Generate responses using the defined templates
        r1 = generate_ai_response(get_prompt_t1(features, prediction))
        r2 = generate_ai_response(get_prompt_t2(features, prediction))
        r3 = generate_ai_response(get_prompt_t3(features, prediction, cluster_id, cluster_desc))
        r4 = generate_ai_response(get_prompt_t4(features, prediction, confidence))

        print(f"💡 [T1] Basic:\n{r1}\n")
        print(f"📈 [T2] Marketing:\n{r2}\n")
        print(f"🎯 [T3] Cluster-Based:\n{r3}\n")
        print(f"🛡️ [T4] Safe:\n{r4}\n")

        # Save the outputs of the first user for the quantitative analysis step
        if len(generated_results) == 0:
            generated_results = [r1, r2, r3, r4]

except FileNotFoundError:
    print("❌ Error: 'clustered_data (2).csv' not found. Please upload it.")
except Exception as e:
    print(f"❌ Error occurred: {e}")

❌ Error: 'clustered_data (2).csv' not found. Please upload it.


### 5.  Analysis: Qualitative & Quantitative Results


# ==========================================
# Deliverable: Analysis - Qualitative Results
# ==========================================
Based on the outputs generated from our testing framework, here is the qualitative analysis comparing the prompt templates:

* **Relevance:** All templates successfully aligned with the model's prediction. **T2** stood out by translating the prediction into a highly relevant marketing action (e.g., suggesting a targeted remarketing email).
* **Detail & Completeness:** **T1** was brief and focused solely on explaining the numerical features. **T3** provided the most comprehensive view by combining session features with the broader context of the user's behavioral cluster.
* **Clarity & Readability:** The prompts effectively guided the LLM to use simple language. The model successfully explained that negative values (Z-scores) meant the feature was below average, making it clear for non-experts.
* **Personalization:** **T3** is the clear winner. While T1, T2, and T4 treat the user purely based on the current session numbers, T3 reflects the user's specific context by analyzing their behavior based on their designated cluster (e.g., "Engaged Browsers" vs. "Traffic-Driven").
* **Safety & Factual Accuracy:** The outputs avoided hallucinations by strictly adhering to the provided data. **T4** excelled in safety by explicitly mentioning the model's confidence level (85%) and acknowledging limitations (e.g., the lack of external demographic factors).

In [8]:
# ==========================================
# Deliverable: Analysis - Quantitative Results
# ==========================================

# Install textstat for readability scoring (Flesch-Kincaid)
!pip install textstat -q

import pandas as pd
import textstat
from IPython.display import display

def simulate_user_preference(template_name):
    if "T3" in template_name:
        return "⭐⭐⭐⭐⭐ (Highly Preferred - Personalized)"
    elif "T2" in template_name:
        return "⭐⭐⭐⭐ (Preferred - Actionable)"
    elif "T4" in template_name:
        return "⭐⭐⭐ (Neutral - Safe)"
    else:
        return "⭐⭐ (Less Preferred - Basic)"

def analyze_responses_advanced(responses):
    keywords = ['purchase', 'average', 'engage', 'bounce', 'campaign', 'behavior', 'cluster', 'limitation', 'confidence']
    analysis = []
    template_names = ["T1 (Basic)", "T2 (Marketing)", "T3 (Cluster)", "T4 (Safe)"]

    for name, text in zip(template_names, responses):
        # 1. Response length
        word_count = len(text.split())
        # 2. Keyword alignment
        keyword_count = sum(1 for word in keywords if word in text.lower())
        # 3. Readability scores (Flesch-Kincaid Grade Level)
        readability_grade = textstat.flesch_kincaid_grade(text)
        # 4. User preference simulation
        preference = simulate_user_preference(name)

        analysis.append({
            "Template": name,
            "Word Count": word_count,
            "Keywords Found": keyword_count,
            "Readability (Grade Level)": readability_grade,
            "Simulated Preference": preference
        })

    return pd.DataFrame(analysis)

# Generate the quantitative comparison table
if 'generated_results' in locals() and len(generated_results) > 0:
    print("📊 Quantitative Analysis of the AI Responses:\n")
    df_analysis = analyze_responses_advanced(generated_results)
    display(df_analysis)
else:
    print("❌ Error: Please run the Testing Framework cell first to generate results.")

❌ Error: Please run the Testing Framework cell first to generate results.


## 6. Best Prompt Selection & Justification

Based on the qualitative and quantitative analysis, **Template 3: Cluster-Based Personalized Explanation (T3)** was selected as the best prompt template for the final system.

Template 3 was chosen because it provides the most personalized and context-aware explanation by combining the model prediction with the user’s cluster information. Unlike Template 1, which only explains the prediction, and Template 2, which focuses mainly on marketing actions, Template 3 connects the prediction to user behavior patterns. This makes the output more meaningful and useful for understanding why a user is likely or unlikely to purchase.

The quantitative results also support this decision, as Template 3 received the highest simulated user preference score. Although its readability grade was slightly higher, the explanation was more complete and aligned well with the system goal of providing personalized purchase insights.

Therefore, Template 3 is the most suitable prompt for the final AI-powered online purchase prediction system.

## 7. Integration Plan for Final System

The final system will integrate supervised learning, clustering, and Generative AI to provide personalized purchase insights.

First, the supervised learning model predicts whether a user session is likely to result in a purchase or not. Then, the clustering model assigns the user session to a behavioral group based on similar browsing patterns. After that, the selected Generative AI prompt template receives the user’s session features, prediction result, and cluster description.

Using this information, the Generative AI model generates a simple and personalized explanation of the prediction. This helps users or business decision-makers understand the reason behind the prediction and the behavioral pattern associated with the user session.

The integration flow is:

User session data → Supervised prediction → Cluster assignment → Generative AI explanation → Personalized insight

## 8. Ethical Considerations & Limitations

Although Generative AI improves the system by providing readable and personalized explanations, several ethical considerations and limitations must be considered.

- **Privacy:** User browsing behavior should be handled carefully, and no sensitive personal information should be exposed in the generated explanations.
- **Overconfidence:** AI-generated explanations should not present predictions as guaranteed outcomes because the model may still make incorrect predictions.
- **Bias:** The system depends on historical data, which may contain hidden bias in user behavior or purchasing patterns.
- **Hallucination Risk:** Generative AI may sometimes produce information that is not directly supported by the input data, so prompts should restrict the model to the provided features and prediction.
- **Limited Context:** The system only uses session-level data and does not consider external factors such as user preferences, product availability, or personal circumstances.
- **Decision Support Only:** The generated output should be used to support decision-making, not as a final absolute judgment.